# E3 — verificación de la demo con pesos reales (PREPARADO, NO EJECUTADO)
Ejecutar en Colab con GPU (Entorno de ejecución → T4). Objetivo: una máscara de punta a punta con SAM 2.1 tiny real y puntos manuales.
**Salida esperada:** `OK e3_demo_smoke` y el archivo `/content/e3_demo_smoke.json` (descargarlo). Opcional: el brazo ajustado, si se sube el `best/payload` de E2.
`HF_TOKEN` se lee de los secretos de Colab (icono de llave); no se imprime ni se guarda.

In [ ]:
import os, subprocess, sys
os.environ['AUTO_PROMPT_EXECUTION_CONTEXT']='google_colab_user_notebook'
subprocess.run('rm -rf /content/auto-prompt-sam2 && git clone -q -b _E.V_ https://github.com/roberto-chavez/auto-prompt-sam2 /content/auto-prompt-sam2',shell=True,check=True)
os.chdir('/content/auto-prompt-sam2'); sys.path.insert(0,'src')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-demo-inference.txt'],check=True)
try:
    from google.colab import userdata; os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')
except Exception as e: print('Sin HF_TOKEN (el SAM 2.1 tiny es público):',type(e).__name__)
import torch; assert torch.cuda.is_available(),'Activar GPU'
print(torch.cuda.get_device_name(0))

Sube un chip Sen1Floods11 de **validación** (p. ej. `Ghana_1033830_S2Hand.tif`, SHA-256 `aebdf646…3c69`) a `/content/chip.tif`, desde tu Drive o con el panel de archivos. No lo subas a Git.

In [ ]:
import json, hashlib, shutil, numpy as np
from pathlib import Path
SRC='/content/drive/MyDrive/auto-prompt-sam2/raw/v1.1/data/flood_events/HandLabeled/S2Hand/Ghana_1033830_S2Hand.tif'  # ajustar si es otra ruta
if not Path('/content/chip.tif').exists():
    from google.colab import drive; drive.mount('/content/drive'); shutil.copy(SRC,'/content/chip.tif')
print('sha256 chip', hashlib.sha256(open('/content/chip.tif','rb').read()).hexdigest())

In [ ]:
from auto_prompt_sam2.demo_config import load_demo_config
from auto_prompt_sam2.demo_backend import DemoBackend
from auto_prompt_sam2.demo_images import load_image, S2_LAYOUT
cfg=load_demo_config(environ={**os.environ,'DEMO_MODE':'inference','DEMO_ALLOW_DOWNLOAD':'1'})
be=DemoBackend(cfg)
img=load_image('/content/chip.tif',S2_LAYOUT,be.params)
print(img.rgb.shape, 'válidos', int(img.valid.sum()))
# Puntos manuales (x,y,etiqueta): ajustar tras mirar img.preview(); 1=agua, 0=fondo. Deben caer en píxeles válidos.
import matplotlib.pyplot as plt; plt.imshow(img.preview()); plt.show()

In [ ]:
POINTS=[(256,256,1),(100,100,0)]   # EDITAR: al menos un positivo sobre agua visible
out=be.segment(img,POINTS,compare=False)
m=out['frozen']['mask']; print('píxeles de máscara:',int(m.sum()),'estado:',out['status'])
assert m.dtype==bool and m.shape==img.valid.shape and m.sum()>0,'La máscara salió vacía: mover los puntos'
from PIL import Image; Image.fromarray((m*255).astype('uint8')).save('/content/e3_demo_mask.png')
json.dump({'points':POINTS,'mask_pixels':int(m.sum()),'selection':str(out['frozen']['selection']),'gpu':torch.cuda.get_device_name(0),
  'chip_sha256':hashlib.sha256(open('/content/chip.tif','rb').read()).hexdigest(),'checkpoint':'facebook/sam2.1-hiera-tiny (verificado por SHA contra report.json)'},open('/content/e3_demo_smoke.json','w'),indent=1)
print('OK e3_demo_smoke')

**Descargar:** `e3_demo_smoke.json` y `e3_demo_mask.png` (panel de archivos → clic derecho → Descargar) y enviarlos al repositorio de evidencia (`docs/project/evidence/e3/`). Sin esos dos archivos la demo sigue sin estar verificada.